# 08 — Tools

@tool/defineTool, auto-injected transfer_call/end_call, dynamic variables, custom tools, schema validation.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises the `tool()` factory and agent tool registration.


### `tool()` factory


In [ ]:
import { tool } from "getpatter";
await cell('tool_decorator', { tier: 1, env }, () => {
  const getWeather = tool({
    name: 'get_weather',
    description: 'Get the current weather for a city.',
    parameters: { city: { type: 'string' }, units: { type: 'string', default: 'celsius' } },
    handler: ({ city, units = 'celsius' }: { city: string; units?: string }) =>
      `Sunny, 22${units[0].toUpperCase()} in ${city}`,
  });
  console.log(`name:  ${getWeather.name}`);
  console.log(`call:  ${getWeather.handler({ city: 'Paris' })}`);
});


### `Tool()` constructor


In [ ]:
import { tool } from "getpatter";
await cell('tool_inline', { tier: 1, env }, () => {
  const searchTool = tool({
    name: 'web_search',
    description: 'Search the web for up-to-date information.',
    parameters: { query: { type: 'string' }, numResults: { type: 'integer', default: 5 } },
    handler: ({ query, numResults = 5 }: { query: string; numResults?: number }) =>
      `Top ${numResults} results for '${query}'`,
  });
  console.log(`name: ${searchTool.name}`);
  console.log(`call: ${searchTool.handler({ query: 'Patter SDK', numResults: 3 })}`);
  if (searchTool.name !== 'web_search') throw new Error('wrong name');
});


### Agent with tools list


In [ ]:
import { Patter, Twilio, OpenAIRealtime, tool } from "getpatter";
await cell('tool_in_agent', { tier: 1, env }, () => {
  const bookTool = tool({
    name: 'book_appointment',
    description: 'Book an appointment.',
    parameters: { date: { type: 'string' }, time: { type: 'string' } },
    handler: ({ date, time }: { date: string; time: string }) => `Booked ${date} ${time}`,
  });
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const agent = p.agent({
    systemPrompt: 'You are a booking assistant.',
    engine: new OpenAIRealtime({ apiKey: 'sk-test' }),
    tools: [bookTool],
  });
  console.log(`Agent tools: ${agent.tools?.map((t: any) => t.name)}`);
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Fires a real tool call mid-call and demonstrates `transfer_call`. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
await cell('live_preflight', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, () => {
  console.log(`  carrier:  Twilio ${env.twilioNumber}  →  ${env.targetNumber}`);
  console.log('  tools:    lookup_order (custom) + transfer_call (auto-injected)');
  console.log(`  webhook:  ${env.publicWebhookUrl || '(ngrok auto-launch)'}`);
});


### Live call with custom tool *(T4)*


In [ ]:
import { Patter, Twilio, OpenAIRealtime, tool } from "getpatter";
await cell('live_tools_call', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, async () => {
  const lookupOrder = tool({
    name: 'lookup_order',
    description: 'Look up order status.',
    parameters: { orderId: { type: 'string' } },
    handler: ({ orderId }: { orderId: string }) => `Order ${orderId}: shipped.`,
  });
  const p = new Patter({
    carrier: new Twilio({ accountSid: env.twilioSid, authToken: env.twilioToken }),
    phoneNumber: env.twilioNumber,
    webhookUrl: env.publicWebhookUrl,
  });
  const agent = p.agent({
    systemPrompt: 'Demo order assistant. Look up order 12345 if asked.',
    engine: new OpenAIRealtime({ apiKey: env.openaiKey }),
    tools: [lookupOrder],
  });
  try {
    await p.call(env.targetNumber, { agent, firstMessage: 'Hi! Ask about order 12345.', ringTimeout: env.maxCallSeconds });
    console.log('✓ Tools call completed');
  } finally {
    await hangupLeftoverCalls(env);
  }
});
